<a href="https://colab.research.google.com/github/GK-BOTZ/Resources/blob/main/Session%20String%20Generate/COLAB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<h1>Telegram Session Generator</h1>

<center><img src='https://telegram.org/img/t_logo.png' height="180" width="180"/></center>

---

### Features
- Generate Pyrogram / Pyrofork Session Strings
- Supports OTP & 2FA
- Auto installs selected library
- Sends session to Saved Messages
- Cleans all session files automatically

---

### Get API Credentials
- https://youtu.be/7LwECAzf8AU

In [ ]:
#@title <center><h3><b>Telegram Session Generator</b></h3></center><br>

#@markdown ---
#@markdown ### Generate Session Securely
#@markdown - Supports Pyrogram / Pyrofork
#@markdown - Sends Session To Saved Messages
#@markdown - Cleans Local Session Files

LIBRARY = "" #@param {type:"string"}
API_ID = "" #@param {type:"string"}
API_HASH = "" #@param {type:"string"}
PHONE_NUMBER = "" #@param {type:"string"}

#@markdown ---

import os, sys, glob, shutil, uuid, subprocess
from importlib.util import find_spec

MAX_RETRY = 3

def run(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True)

def install(pkg):

    module = pkg.replace('-', '_')

    if find_spec(module) is None:

        print(f'Installing {pkg}...')

        x = run(f'pip install -q {pkg}')

        if x.returncode != 0:
            print(x.stderr)
            sys.exit(1)

def ask(text, value):

    if value.strip():
        return value.strip()

    for _ in range(MAX_RETRY):

        x = input(text).strip()

        if x:
            return x

        print('Invalid Value')

    sys.exit(1)

LIBRARY = ask(
    'Library (pyrogram/pyrofork etc): ',
    LIBRARY
).lower()

API_ID = ask(
    'API ID: ',
    API_ID
)

API_HASH = ask(
    'API HASH: ',
    API_HASH
)

PHONE_NUMBER = ask(
    'PHONE NUMBER: ',
    PHONE_NUMBER
)

install(LIBRARY)
install('tgcrypto')

try:

    from pyrogram import Client

    from pyrogram.errors import (
        SessionPasswordNeeded,
        PhoneCodeInvalid,
        PasswordHashInvalid
    )

except Exception as e:

    print(e)

    sys.exit(1)

for i in glob.glob('USER*') + glob.glob('*.session*'):

    try:
        shutil.rmtree(i) if os.path.isdir(i) else os.remove(i)

    except Exception as e:
        print(e)

SESSION = f'USER_{uuid.uuid4().hex}'

app = Client(
    SESSION,
    api_id=int(API_ID),
    api_hash=API_HASH,
    phone_number=PHONE_NUMBER,
    in_memory=True
)

async def main():

    try:

        await app.connect()

        sent = None

        for _ in range(MAX_RETRY):

            try:

                sent = await app.send_code(
                    PHONE_NUMBER
                )

                break

            except Exception as e:

                print(e)

        if not sent:
            raise Exception(
                'Failed Sending OTP'
            )

        ok = False

        for _ in range(MAX_RETRY):

            try:

                otp = input(
                    'OTP: '
                ).strip()

                await app.sign_in(
                    PHONE_NUMBER,
                    sent.phone_code_hash,
                    otp
                )

                ok = True

                break

            except SessionPasswordNeeded:

                break

            except PhoneCodeInvalid:

                print('Invalid OTP')

            except Exception as e:

                print(e)

        if not ok:

            for _ in range(MAX_RETRY):

                try:

                    pw = input(
                        '2FA Password: '
                    ).strip()

                    await app.check_password(
                        password=pw
                    )

                    ok = True

                    break

                except PasswordHashInvalid:

                    print(
                        'Wrong Password'
                    )

                except Exception as e:

                    print(e)

        if not ok:

            raise Exception(
                'Login Failed'
            )

        session = await app.export_session_string()

        try:

            text = (
                f'**Tʜɪs Is Yᴏᴜʀ '
                f'__{LIBRARY}__ '
                f'Sᴛʀɪɴɢ Sᴇssɪᴏɴ**\n\n'
                f'`{session}`\n\n'
                f'⚠️ **Dᴏɴ\'ᴛ Sʜᴀʀᴇ '
                f'Tʜɪs Wɪᴛʜ Aɴʏᴏɴᴇ**'
            )

            await app.send_message(
                'me',
                text
            )

        except Exception as e:

            print(e)

        me = await app.get_me()

        print(
            f'\nSession Generated Successfully '
            f'for @{me.username or me.first_name}'
        )

    except Exception as e:

        print(f'\n{e}')

    finally:

        try:

            app.storage.SESSION_STRING = None

            await app.disconnect()

        except Exception as e:

            print(e)

        for i in glob.glob('USER*') + glob.glob('*.session*'):

            try:

                shutil.rmtree(i) if os.path.isdir(i) else os.remove(i)

            except Exception as e:

                print(e)

        try:

            print(f'\nUninstalling {LIBRARY} and tgcrypto...\n')

            run(f'pip uninstall -y {LIBRARY} tgcrypto')

        except Exception as e:

            print(e)

        print('\nSession Files Cleaned')

await main()